Jupyter notebook to run through the getting started section of the icet Python library (https://icet.materialsmodeling.org/)

## Constructing the cluster expansion, Ag-Pd

In [ ]:
from ase.db import connect
from icet import ClusterSpace, StructureContainer, ClusterExpansion
from trainstation import CrossValidationEstimator

In [ ]:
db = connect('../data/reference_data.db')
primitive_structure = db.get(id=1).toatoms()  # primitive structure

To build the cluster expansion we first create a `ClusterSpace` from a prototype structure. This also requires cutoffs and the chemical elements allowed on each lattice site. The cutoffs set the maximum pairwise distance between atoms that can belong to the same cluster, given separately for each cluster order (pairs, triplets, quadruplets). Distances here are measured in Angstroms (Å), the standard length unit in atomistic modelling (1 Å = 10⁻¹⁰ m).

See https://icet.materialsmodeling.org/get_started/construct_cluster_expansion.html for more detail.

In [ ]:
cs = ClusterSpace(structure=primitive_structure,
                  cutoffs=[13.5, 6.5, 6.0], # the cut-offs in question, 13.5 for pairs, 6.5 for triplets, 6 for quadruplets.
                  chemical_symbols=['Ag', 'Pd']) # the allowed elements
display(cs)

Next we initialise a `StructureContainer`. We only attach one property (`mixing_energy`) here, but a container can hold several target properties per structure.

**`ClusterSpace` vs `StructureContainer`:**

```
ClusterSpace       = the basis. Given a prototype lattice, cutoffs, and allowed
                     elements, it enumerates all symmetrically distinct clusters
                     (pairs, triplets, ...). Defines *what* to measure, no data yet.

StructureContainer = the data. For each structure in the DB, projects it onto
                     the ClusterSpace basis -> feature vector, paired with target
                     property (e.g. mixing_energy).

Polynomial regression analogy:
  Choice of degree (e.g. up to x^3)   -> ClusterSpace (cutoffs + allowed elements)
  [1, x, x^2, x^3] for a given x      -> cluster vector (82 numbers here) for a crystal
  Design matrix X (stacked rows)      -> StructureContainer
  Coefficients c_i from least-squares -> ECIs from ClusterExpansion fit
  Final model: y ≈ sum(c_i * x^i)     -> energy ≈ X @ ECI
```

Cutoffs ↔ polynomial degree: both set how expressive the feature space is. Looser cutoffs / higher degree = more features, richer model, more overfit risk. It's a modelling choice about the basis, **not** a filter on the data.

The basis and data are kept separate because the basis is reusable: same `ClusterSpace`, swap datasets or add more target properties.

In [ ]:
sc = StructureContainer(cluster_space=cs)
for row in db.select():
    sc.add_structure(structure=row.toatoms(),
                     user_tag=row.tag,
                     properties={'mixing_energy': row.mixing_energy})
display(sc)

We can now fit the effective cluster interactions (ECIs) against the target mixing energies. We use cross-validation to choose a fit that generalises well beyond the training structures, then call `train` to produce the final coefficients.

In [ ]:
opt = CrossValidationEstimator(
    fit_data=sc.get_fit_data(key='mixing_energy'), fit_method='ardr')
opt.validate()
opt.train()
print(opt)

Roughly half of the 82 parameters are driven to zero. This is the effect of `ardr` (Automatic Relevance Determination Regression), a Bayesian sparse regression method that prunes unimportant features, somewhat similar to lasso, which keeps the model compact and helps generalisation.

Last step: attach the fitted parameters to the `ClusterSpace` to form a `ClusterExpansion`. On their own the parameters are just a vector of numbers. Pairing them with the cluster space is what lets us evaluate energies for new structures.

In [ ]:
ce = ClusterExpansion(
    cluster_space=cs, parameters=opt.parameters, metadata=opt.summary)
display(ce)
ce.write('mixing_energy.ce')

## Analysing the Cluster Expansion

To gauge performance, we compare the cluster expansion's predicted mixing energies against the reference (DFT) mixing energies for the training structures.

In [ ]:
ce = ClusterExpansion.read('mixing_energy.ce')
data = {'concentration': [], 'reference_energy': [], 'predicted_energy': []}
db = connect('../data/reference_data.db')
for row in db.select('natoms<=6'):
    data['concentration'].append(row.concentration)
    # the factor of 1e3 serves to convert from eV/atom to meV/atom
    data['reference_energy'].append(1e3 * row.mixing_energy)
    data['predicted_energy'].append(1e3 * ce.predict(row.toatoms()))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(4, 3))
ax.set_xlabel(r'Pd concentration')
ax.set_ylabel(r'Mixing energy (meV/atom)')
ax.set_xlim([0, 1])
ax.set_ylim([-69, 15])
ax.scatter(data['concentration'], data['reference_energy'],
           marker='o', label='reference')
ax.scatter(data['concentration'], data['predicted_energy'],
           marker='x', label='CE prediction')
plt.savefig('mixing_energy_comparison.png', bbox_inches='tight')

Predicted (orange crosses) and target (blue circles) mixing energies versus concentration for the structures used in the construction of the cluster expansion

Now we want to use our cluster expansion to predict mixing energies for a larger set of structures, which we obtain by enumeration.

In [ ]:
import matplotlib.pyplot as plt
from numpy import array
from icet import ClusterExpansion
from icet.tools import ConvexHull, enumerate_structures

In [ ]:
ce = ClusterExpansion.read('mixing_energy.ce')
species = ['Ag', 'Pd']
data = {'concentration': [], 'mixing_energy': []}
structures = []
cluster_space = ce.get_cluster_space_copy()
chemical_symbols = cluster_space.chemical_symbols
primitive_structure = cluster_space.primitive_structure
for structure in enumerate_structures(structure=primitive_structure,
                                      sizes=range(1, 13),
                                      chemical_symbols=chemical_symbols):
    conc = structure.symbols.count('Pd') / len(structure)
    data['concentration'].append(conc)
    data['mixing_energy'].append(ce.predict(structure))
    structures.append(structure)
print('Predicted energies for {} structures'.format(len(structures)))

Extract the convex hull of mixing energy vs. concentration, its vertices are the ground-state structures at each composition.

In [ ]:
hull = ConvexHull(data['concentration'], data['mixing_energy'])

Now plot that

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
ax.set_xlabel(r'Pd concentration')
ax.set_ylabel(r'Mixing energy (meV/atom)')
ax.set_xlim([0, 1])
ax.set_ylim([-69, 15])
ax.scatter(data['concentration'], 1e3 * array(data['mixing_energy']),
           marker='x')
ax.plot(hull.concentrations, 1e3 * hull.energies, '-o', color='green')
plt.savefig('mixing_energy_predicted.png', bbox_inches='tight')

Now filter for low energy configurations

In [ ]:
tol = 0.0005
low_energy_structures = hull.extract_low_energy_structures(
    data['concentration'], data['mixing_energy'], tol)
print('Found {} structures within {} meV/atom of the convex hull'.
      format(len(low_energy_structures), 1e3 * tol))

Now inspect how the ECIs vary with cluster radius and order. Physically meaningful fits tend to show interactions decaying with distance and with increasing order; if they don't, the cutoffs or fit method may need revisiting.

In [ ]:
import numpy as np

ce = ClusterExpansion.read('mixing_energy.ce')
df_ecis = ce.to_dataframe()

In [ ]:
fig, axs = plt.subplots(1, 3, sharey=True, figsize=(7.5, 3))
for k, order in enumerate(ce.orders):
    df_order = df_ecis.loc[df_ecis['order'] == order]
    if k < 2 or k > 4:
        continue
    ax = axs[k - 2]
    ax.set_ylim((-1, 8))
    ax.set_xlabel(r'Cluster radius (Å)')
    if order == 2:
        ax.set_xlim((1.2, 4.2))
        ax.set_ylabel(r'Effective cluster interaction (meV)')
    if order == 3:
        ax.set_xlim((1.5, 3.9))
    if order == 4:
        ax.set_xlim((1.5, 3.9))
        ax.text(0.05, 0.55, 'zerolet: {:.1f} meV'
                .format(1e3 * df_ecis.eci.iloc[0]),
                transform=ax.transAxes)
        ax.text(0.05, 0.45, 'singlet: {:.1f} meV'
                .format(1e3 * df_ecis.eci.iloc[1]),
                transform=ax.transAxes)
    ax.plot([0, 5], [0, 0], color='black')
    ax.bar(df_order.radius, 1e3 * df_order.eci, width=0.05)
    ax.scatter(df_order.radius, len(df_order) * [-0.7],
               marker='o', s=2.0)
    ax.text(0.05, 0.91, 'order: {}'.format(order),
            transform=ax.transAxes)
    ax.text(0.05, 0.81, '#parameters: {}'.format(len(df_order)),
            transform=ax.transAxes,)
    ax.text(0.05, 0.71, '#non-zero params: {}'
            .format(np.count_nonzero(df_order.eci)),
            transform=ax.transAxes,)
plt.savefig('ecis.png', bbox_inches='tight')

## Sampling the Cluster Expansion

Now we have our cluster expansion, we can use the built-in MCMC sampler to draw configurations from it.

In [ ]:
from ase.build import make_supercell
from icet import ClusterExpansion
from mchammer.calculators import ClusterExpansionCalculator
from mchammer.ensembles import SemiGrandCanonicalEnsemble, VCSGCEnsemble
import numpy as np
from os import mkdir

**Supercell.** A crystal's *primitive cell* is the smallest repeating unit that tiles the lattice under periodic boundary conditions — for FCC Ag that's a single atom. A *supercell* is a larger box built by tiling the primitive cell (here 108 atoms), which becomes the MC simulation box. We need this because MC samples configurations by swapping species across many sites — one atom has nothing to flip, and finite-size effects shrink as the box grows.

The `ClusterExpansionCalculator` wraps the CE + supercell so the MC ensembles can evaluate energies on it. The self-interaction warning appears because the 13.5 Å pair cutoff exceeds half the supercell edge (clusters wrap onto their own periodic image) — tolerable for a tutorial, but production runs want a larger cell.

In [ ]:
ce = ClusterExpansion.read('mixing_energy.ce')
structure = make_supercell(ce.get_cluster_space_copy().primitive_structure,
                           3 * np.array([[-1, 1, 1],
                                         [1, -1, 1],
                                         [1, 1, -1]]))
calculator = ClusterExpansionCalculator(structure, ce)

**Semi-Grand Canonical (SGC) ensemble.** Atom count is fixed but composition fluctuates: each trial step flips a site's species and accepts/rejects based on energy change plus a chemical-potential penalty Δμ = μ_Pd − μ_Ag. Sweeping Δμ at fixed T traces out composition vs. chemical potential, which is the thermodynamic response we want for phase behaviour. We run at T = 900 K (expected disordered) and T = 300 K (expected ordered) to see how ordering changes with temperature.

In [ ]:
# Make sure output directory exists
output_directory = 'monte_carlo_data'
try:
    mkdir(output_directory)
except FileExistsError:
    pass
for temperature in [900, 300]:
    # Evolve configuration through the entire composition range
    for dmu in np.arange(-0.7, 0.51, 0.05):
        # Initialize MC ensemble
        fname = f'{output_directory}/sgc-T{temperature}-dmu{dmu:+.3f}.dc'
        mc = SemiGrandCanonicalEnsemble(
            structure=structure,
            calculator=calculator,
            temperature=temperature,
            dc_filename=fname,
            chemical_potentials={'Ag': 0, 'Pd': dmu})

        mc.run(number_of_trial_steps=len(structure) * 30)
        structure = mc.structure

**Variance-Constrained SGC (VCSGC).** Same spirit as SGC, but swaps the linear Δμ term for a *quadratic* constraint that softly pins the average concentration near a target value (controlled by `phi` ∈ roughly [−2, 0]) with strength `kappa`. This fixes SGC's main weakness: inside two-phase regions SGC jumps discontinuously between compositions, so you can't sample intermediate concentrations. VCSGC can, which makes it the better tool for mapping miscibility gaps and ordering transitions.

In [ ]:
# Make sure output directory exists
output_directory = 'monte_carlo_data'
try:
    mkdir(output_directory)
except FileExistsError:
    pass
for temperature in [900, 300]:
    # Evolve configuration through the entire composition range
    for phi in np.arange(-2.1, 0.11, 0.08):
        # Initialize MC ensemble
        fname = f'{output_directory}/vcsgc-T{temperature}-phi{phi:+.3f}.dc'
        mc = VCSGCEnsemble(
            structure=structure,
            calculator=calculator,
            temperature=temperature,
            dc_filename=fname,
            phis={'Pd': phi},
            kappa=200)

        mc.run(number_of_trial_steps=len(structure) * 30)
        structure = mc.structure

We now want to analyse our simulations.

In [ ]:
import pandas as pd
from glob import glob
from mchammer import DataContainer

Collect data from simulations and write to csv format.

In [ ]:
for ensemble in ['sgc', 'vcsgc']:
    data = []
    for filename in glob('monte_carlo_data/{}-*.dc'.format(ensemble)):
        dc = DataContainer.read(filename)
        data_row = dc.ensemble_parameters
        data_row['filename'] = filename
        n_atoms = data_row['n_atoms']

        equilibration = 5 * n_atoms

        stats = dc.analyze_data('Pd_count', start=equilibration)
        data_row['Pd_concentration'] = stats['mean'] / n_atoms
        data_row['Pd_concentration_error'] = stats['error_estimate'] / n_atoms

        stats = dc.analyze_data('potential', start=equilibration)
        data_row['mixing_energy'] = stats['mean'] / n_atoms
        data_row['mixing_energy_error'] = stats['error_estimate'] / n_atoms

        data_row['acceptance_ratio'] = \
            dc.get_average('acceptance_ratio', start=equilibration)
        if ensemble == 'sgc':
            data_row['free_energy_derivative'] = \
                dc.ensemble_parameters['mu_Pd'] - \
                dc.ensemble_parameters['mu_Ag']
        elif ensemble == 'vcsgc':
            data_row['free_energy_derivative'] = \
                dc.get_average('free_energy_derivative_Pd', start=equilibration)

        data.append(data_row)
    
    df = pd.DataFrame(data)
    df.to_csv('monte-carlo-{}.csv'.format(ensemble), sep='\t')

In [ ]:
dfs = {}
dfs['sgc'] = pd.read_csv('monte-carlo-sgc.csv', delimiter='\t')
dfs['vcsgc'] = pd.read_csv('monte-carlo-vcsgc.csv', delimiter='\t')


# step 1: Plot free energy derivatives
colors = {300: '#D62728',
          900: '#1F77B4'}
linewidths = {'sgc': 3, 'vcsgc': 1}
alphas = {'sgc': 0.5, 'vcsgc': 1.0}
fig, ax = plt.subplots(figsize=(4, 3.5))
for ensemble, df in dfs.items():
    for T in sorted(df.temperature.unique()):
        df_T = df.loc[df['temperature'] == T].sort_values('Pd_concentration')
        ax.plot(df_T['Pd_concentration'],
                1e3 * df_T['free_energy_derivative'],
                marker='o', markersize=2.5,
                label='{}, {} K'.format(ensemble, T),
                color=colors[T],
                linewidth=linewidths[ensemble], alpha=alphas[ensemble])
ax.set_xlabel('Pd concentration')
ax.set_ylabel('Free energy derivative (meV/atom)')
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-600, 500])
ax.legend()
plt.savefig('free_energy_derivative.png', bbox_inches='tight')

In [ ]:
df = dfs['sgc']
fig, ax = plt.subplots(figsize=(4, 3.5))
for T in sorted(df.temperature.unique()):
    df_T = df.loc[df['temperature'] == T].sort_values('Pd_concentration')
    e_mix = 1e3 * df_T['mixing_energy']
    e_mix_error = 1e3 * df_T['mixing_energy_error']
    ax.plot(df_T['Pd_concentration'], e_mix,
            marker='o', markersize=2.5, label='{} K'.format(T),
            color=colors[T])
    # Plot error estimate
    ax.fill_between(df_T['Pd_concentration'],
                    e_mix + e_mix_error, e_mix - e_mix_error,
                    color=colors[T], alpha=0.4)
ax.set_xlabel('Pd concentration')
ax.set_ylabel('Mixing energy (meV/atom)')
ax.set_xlim([-0.02, 1.02])
ax.legend()
plt.savefig('mixing_energy_sgc.png', bbox_inches='tight')

In [ ]:
df = dfs['sgc']
fig, ax = plt.subplots(figsize=(4, 3.5))
for T in sorted(df.temperature.unique()):
    df_T = df.loc[df['temperature'] == T].sort_values('Pd_concentration')
    ax.plot(df_T['Pd_concentration'], df_T['acceptance_ratio'],
            marker='o', markersize=2.5, label='{} K'.format(T),
            color=colors[T])
ax.set_xlabel('Pd concentration')
ax.set_ylabel('Acceptance ratio')
ax.set_xlim([-0.02, 1.02])
ax.legend()
plt.savefig('acceptance_ratio_sgc.png', bbox_inches='tight')

## Effective Sample Size (ESS)

MCMC samples are autocorrelated, so the *number* of recorded entries overstates how much information a chain actually carries. ESS rescales: `ESS = N / τ`, where `τ` is the integrated autocorrelation time.

**Unit subtlety.** `mchammer`'s `analyze_data` reports `correlation_length` in *trial steps*, not in *recorded entries* (it multiplies the lag-in-samples by the recording interval, see `mchammer/data_containers/data_container.py`). The recording interval here defaults to `len(structure)` = 108 trial steps per entry, so `correlation_length` always comes out as a multiple of 108. To compute ESS we either work entirely in trial steps (`ESS = total_trial_steps / τ_trialsteps`) or convert `τ` back to samples (`τ_samples = τ_trialsteps / interval`); both give the same answer.

**How to read it.** `ess_per_sample = ESS / N` ∈ (0, 1]. Values near 1 mean nearly independent draws; values near 0 mean the chain is stuck. Drops in ESS coincide with regions where the sampler struggles — typically inside two-phase regions for SGC, where the chain flips slowly between compositions. Constant series (chain fully stuck on one configuration) get `τ = ∞` and ESS = 0 by construction.

In [ ]:
from glob import glob
from mchammer import DataContainer

ess_records = []
for ensemble in ['sgc', 'vcsgc']:
    for fn in glob(f'monte_carlo_data/{ensemble}-*.dc'):
        dc = DataContainer.read(fn)
        params = dict(dc.ensemble_parameters)
        n_atoms = params['n_atoms']
        eq = 5 * n_atoms

        mctrials = dc.get('mctrial', start=eq)
        if len(mctrials) < 2:
            continue
        interval = int(mctrials[1] - mctrials[0])              # trial steps per recorded entry
        n_trials_post_eq = int(mctrials[-1] - mctrials[0]) + interval

        pd_series = dc.get('Pd_count', start=eq)
        pd_concentration = float(np.mean(pd_series)) / n_atoms

        for obs in ('potential', 'Pd_count'):
            series = dc.get(obs, start=eq)
            n_samples = len(series)
            if np.std(series) == 0:                            # chain fully stuck → tau = inf
                tau, ess, ess_per_sample = np.inf, 0.0, 0.0
            else:
                stats = dc.analyze_data(obs, start=eq)
                tau = stats.get('correlation_length', np.nan)  # in TRIAL STEPS
                if np.isfinite(tau) and tau > 0:
                    ess = n_trials_post_eq / tau               # both in trial steps
                    ess_per_sample = ess / n_samples
                else:
                    ess = ess_per_sample = np.nan

            ess_records.append({
                'ensemble': ensemble,
                'temperature': params['temperature'],
                'control': params.get('mu_Pd', params.get('phi_Pd', np.nan)),
                'Pd_concentration': pd_concentration,
                'observable': obs,
                'n_samples': n_samples,
                'interval_trial_steps': interval,
                'tau_trial_steps': tau,
                'tau_samples': tau / interval if np.isfinite(tau) else np.inf,
                'ess': ess,
                'ess_per_sample': ess_per_sample,
            })

ess_df = pd.DataFrame(ess_records).sort_values(
    ['ensemble', 'observable', 'temperature', 'Pd_concentration'])

summary = (ess_df
           .groupby(['ensemble', 'observable', 'temperature'])
           [['tau_samples', 'ess', 'ess_per_sample']]
           .agg(['median', 'min', 'max'])
           .round(2))
display(summary)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 6.5), sharex=True, sharey='row')
observables = ['potential', 'Pd_count']
ensembles = ['sgc', 'vcsgc']

for i, obs in enumerate(observables):
    for j, ensemble in enumerate(ensembles):
        ax = axes[i, j]
        sub = ess_df[(ess_df['ensemble'] == ensemble)
                     & (ess_df['observable'] == obs)]
        for T in sorted(sub['temperature'].unique()):
            sT = sub[sub['temperature'] == T].sort_values('Pd_concentration')
            ax.plot(sT['Pd_concentration'], sT['ess_per_sample'],
                    marker='o', markersize=3, color=colors[T],
                    label=f'{T} K')
        ax.set_yscale('log')
        ax.set_ylim(1e-3, 1.2)
        ax.axhline(1.0, color='k', lw=0.5, ls=':')
        if i == 0:
            ax.set_title(ensemble.upper())
        if i == len(observables) - 1:
            ax.set_xlabel('Pd concentration')
        if j == 0:
            ax.set_ylabel(f'ESS / N  ({obs})')
        if i == 0 and j == 0:
            ax.legend(frameon=False)
fig.tight_layout()
plt.savefig('ess_per_sample.png', bbox_inches='tight')